<a href="https://colab.research.google.com/github/suryaph971/Langchain/blob/main/2_A_PineCone_DeepDive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.9/291.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


In [1]:
pip install -q pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 15.2 MB/s eta 0:00:00


In [8]:
from dotenv import load_dotenv,find_dotenv
load_dotenv(find_dotenv(),override=True)

True

In [10]:
from pinecone import ServerlessSpec,PodSpec,Pinecone
pc = Pinecone()

#list indexes -- Specialize data structure which optimizes the search and retrieval of datapoints based on vector representations
pc.list_indexes()

[
    {
        "name": "sampleindex",
        "metric": "dotproduct",
        "host": "sampleindex-ppqt32v.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "sparse",
        "dimension": null,
        "deletion_protection": "disabled",
        "tags": null,
        "embed": {
            "model": "pinecone-sparse-english-v0",
            "field_map": {
                "text": "text"
            },
            "metric": "dotproduct",
            "write_parameters": {
                "input_type": "passage",
                "max_tokens_per_sequence": 512.0,
                "return_tokens": false,
                "truncate": "END"
            },
            "read_parameters": {
                "input_type": "query",
                "max_tokens_per_sequence"

In [11]:
index_name='sampleindex'
pc.describe_index(index_name)

{
    "name": "sampleindex",
    "metric": "dotproduct",
    "host": "sampleindex-ppqt32v.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "sparse",
    "dimension": null,
    "deletion_protection": "disabled",
    "tags": null,
    "embed": {
        "model": "pinecone-sparse-english-v0",
        "field_map": {
            "text": "text"
        },
        "metric": "dotproduct",
        "write_parameters": {
            "input_type": "passage",
            "max_tokens_per_sequence": 512.0,
            "return_tokens": false,
            "truncate": "END"
        },
        "read_parameters": {
            "input_type": "query",
            "max_tokens_per_sequence": 512.0,
            "return_tokens": false,
            "truncate": "END"
        },
        "vector_type": "sparse"
    }
}

In [12]:
pc.list_indexes().names()

['sampleindex']

In [13]:
if index_name in pc.list_indexes().names():
    print(f"Deleting Index :: {index_name}")
    pc.delete_index(index_name)
    print("Done")
else:
    print(f"{index_name} not found")

Deleting Index :: sampleindex
Done


In [15]:
#creating index

index_name='langchain'
if index_name not in pc.list_indexes().names():
    print("Creating Index")
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    print('index ceated')
else:
    print(f"{index_name} already exists")

Creating Index
index ceated


In [16]:
pc.list_indexes().names()

['langchain']

In [19]:
index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

#Working with Vectors


In [21]:
#inserting vectors
import random
vectors=[[random.random() for _ in range(1536)] for v in range(5)]
ids = list('abcde')

index_name='langchain'
index = pc.Index(index_name)
index.upsert(vectors=zip(ids,vectors))

{'upserted_count': 5}

In [23]:
#updating vectors
index.upsert(vectors=[('c',[0.5]*1536)])

{'upserted_count': 1}

In [24]:
#fetching vectors
index.fetch(ids=['a','b'])

FetchResponse(namespace='', vectors={'a': Vector(id='a', values=[0.0524303541, 0.580310464, 0.218542308, 0.277380228, 0.110312201, 0.982456326, 0.649011, 0.222255334, 0.800085187, 0.00239716913, 0.122690551, 0.792381346, 0.734886408, 0.901521742, 0.90266192, 0.261886299, 0.579571664, 0.869546294, 0.593859732, 0.970717251, 0.941109836, 0.982089579, 0.826632082, 0.029925799, 0.813260913, 0.690540791, 0.0393258892, 0.890218616, 0.142562374, 0.139003083, 0.0477169082, 0.512039542, 0.632663846, 0.228886008, 0.120811082, 0.167418271, 0.881502271, 0.985884905, 0.211235955, 0.0991587639, 0.427360177, 0.6206249, 0.540094793, 0.000918819103, 0.20176211, 0.823978603, 0.571172357, 0.294893205, 0.292063296, 0.949453115, 0.731641471, 0.755395353, 0.27437222, 0.537032783, 0.644057631, 0.344479114, 0.3630023, 0.408046961, 0.46300447, 0.259819716, 0.823970139, 0.949974895, 0.605090499, 0.00432442175, 0.85553354, 0.615592837, 0.179375455, 0.833232939, 0.579441845, 0.0118288957, 0.643099308, 0.869357705,

In [25]:
#delete ids

index.delete(ids=['c','d'])

{}

In [26]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 3}},
 'total_vector_count': 3,
 'vector_type': 'dense'}

In [27]:
index.fetch(ids=['x'])

FetchResponse(namespace='', vectors={}, usage={'read_units': 1})

In [28]:
#querying the vectors

query_vector = [random.random() for _ in range(1536)]


In [29]:
index.query(
    vector=query_vector,
    top_k=3,
    include_values=False
)

{'matches': [{'id': 'a', 'score': 0.747309327, 'values': []},
             {'id': 'e', 'score': 0.744994, 'values': []},
             {'id': 'b', 'score': 0.731098771, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

#Namespaces

In [30]:
index = pc.Index('langchain')
import random
vectors = [[random.random() for _ in range(1536)] for v in range(5)]
ids = list('abcde')
index.upsert(vectors=zip(ids,vectors))

{'upserted_count': 5}

In [31]:
#partition the index into namespace
#creating  a new space
vectors = [[random.random() for _ in range(1536)] for v in range(3)]
ids = list('xyz')
index.upsert(vectors = zip(ids,vectors),namespace='first_namespace')


{'upserted_count': 3}

In [32]:
vectors = [[random.random() for _ in range(1536)] for v in range(2)]
ids = list('qp')
index.upsert(vectors=zip(ids,vectors),namespace='Second_namespace')


{'upserted_count': 2}

In [33]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5},
                'Second_namespace': {'vector_count': 2},
                'first_namespace': {'vector_count': 3}},
 'total_vector_count': 10,
 'vector_type': 'dense'}

In [34]:
index.fetch(ids=['x'])

FetchResponse(namespace='', vectors={}, usage={'read_units': 1})

In [35]:
index.fetch(ids=['x'],namespace='first_namespace')


FetchResponse(namespace='first_namespace', vectors={'x': Vector(id='x', values=[0.88664639, 0.804500222, 0.409654796, 0.0503381938, 0.854163289, 0.859624505, 0.953165591, 0.383194923, 0.274880081, 0.947249115, 0.697433114, 0.0950353, 0.789266586, 0.51914221, 0.303787231, 0.9440732, 0.362683624, 0.258004904, 0.30531022, 0.309266865, 0.987643361, 0.369035542, 0.95267719, 0.992168069, 0.861616075, 0.234674394, 0.9031021, 0.765414536, 0.0698897317, 0.240436092, 0.377900511, 0.269218385, 0.114375763, 0.915998399, 0.64001441, 0.110186458, 0.0107631227, 0.103503652, 0.792825818, 0.728994787, 0.711534262, 0.987401545, 0.962795913, 0.150446758, 0.1518341, 0.553457379, 0.466559261, 0.179494724, 0.959145427, 0.060258735, 0.15328677, 0.610473037, 0.763574481, 0.955201924, 0.556735873, 0.175643623, 0.742050469, 0.926840305, 0.37571004, 0.407336444, 0.513077676, 0.388651818, 0.0310894344, 0.375767022, 0.226019874, 0.268536299, 0.0747569427, 0.640553653, 0.259494096, 0.175977901, 0.371420175, 0.51883

In [36]:
index.delete(ids=['q'],namespace='Second_namespace')

{}

In [37]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5},
                'Second_namespace': {'vector_count': 1},
                'first_namespace': {'vector_count': 3}},
 'total_vector_count': 9,
 'vector_type': 'dense'}